# SVM Data Preparation — Shared Aggregated MFCC Cache

This notebook is the **only SVM notebook that scans audio files, creates/reuses the frozen split, extracts MFCCs, aggregates them into 80-dimensional mean+standard-deviation vectors, and fits the `StandardScaler`**.

Run this notebook once before the SVM baseline/trial notebooks. All later SVM notebooks load the same cached arrays and scaler from `outputs/svm/cache`.


## 1. Environment Setup

The project root defaults to the INM701 Google Drive folder in Colab. Set `INTRO_AI_PROJECT_ROOT` to override it.


In [ ]:
import os
from pathlib import Path

# Purpose: Mounts Google Drive when this notebook is running in Google Colab.
# Why this exists: the shared SVM feature cache, manifests, configs, models,
# figures, and metric outputs may live under /content/drive/MyDrive in Colab.
# A /content/drive path is only reliable after Google Drive has actually been mounted.
# This cell is kept separate and early so the rest of the notebook can resolve paths safely.

COLAB_DRIVE_MOUNT_POINT = Path("/content/drive")
EXPECTED_COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")

try:
    from google.colab import drive

    drive.mount("/content/drive")
    os.environ.setdefault(
        "INTRO_AI_PROJECT_ROOT",
        str(EXPECTED_COLAB_PROJECT_ROOT),
    )
    print("Google Colab detected. Google Drive mounted.")
except Exception as exc:
    print("Google Colab Drive mount skipped. This is expected outside Colab.")
    print("Mount skip reason:", exc)

print("INTRO_AI_PROJECT_ROOT:", os.environ.get("INTRO_AI_PROJECT_ROOT", "not set"))


In [1]:
import importlib.util
import subprocess
import sys

required_packages = [
    ("librosa", "librosa"),
    ("soundfile", "soundfile"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("sklearn", "scikit-learn"),
    ("joblib", "joblib"),
]
missing = [pip_name for import_name, pip_name in required_packages if importlib.util.find_spec(import_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Required packages are already installed.")


Required packages are already installed.


## 2. Imports, Configuration and Paths


In [2]:
import json
import os
import random
import shutil
from pathlib import Path

import joblib
import librosa
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

SAMPLE_RATE = 22050
FIXED_DURATION_SECONDS = 5.0
N_MFCC = 40
N_FFT = 1024
HOP_LENGTH = int(round(SAMPLE_RATE * 0.010))
WIN_LENGTH = int(round(SAMPLE_RATE * 0.025))
TARGET_SAMPLES = int(round(SAMPLE_RATE * FIXED_DURATION_SECONDS))
AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3", ".ogg", ".m4a"}
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}
FINAL_SPLIT_VERSION = "mlp_final_split_2026_08_24_v1"
REUSE_MLP_SPLIT = True
RUN_FULL_SVM_DATA_PREPARATION = True
MAX_FILES_PER_CLASS = None


def mount_drive_if_colab():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("Google Drive mounted.")
    except Exception:
        print("Google Drive mount skipped (expected outside Colab).")


mount_drive_if_colab()


def resolve_project_root():
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Data").exists() or (candidate / "Datasets").exists():
            return candidate
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


def first_existing_path(candidates):
    expanded = [Path(item).expanduser() for item in candidates if item not in [None, ""]]
    for path in expanded:
        if path.exists():
            return path
    return expanded[0]


PROJECT_ROOT = resolve_project_root()
DATA_ROOTS = [PROJECT_ROOT / "Data", PROJECT_ROOT / "Datasets"]
SYNTHETIC_AUDIO_DIR = first_existing_path([
    os.environ.get("MLAAD_SYNTHETIC_AUDIO_DIR", ""),
    *[root / "MLAAD_10pct" / "fake" for root in DATA_ROOTS],
    *[root / "MLAAD_10pct" for root in DATA_ROOTS],
    *[root / "Unprocessed" / "MLAAD_10pct" / "fake" for root in DATA_ROOTS],
    *[root / "Unprocessed" / "MLAAD_10pct" for root in DATA_ROOTS],
])
BONA_FIDE_AUDIO_DIR = first_existing_path([
    os.environ.get("MLAAD_BONA_FIDE_AUDIO_DIR", ""),
    *[root / "genuine_audio" for root in DATA_ROOTS],
    *[root / "bona_fide" for root in DATA_ROOTS],
    *[root / "M_AILABS_bona_fide_subset" for root in DATA_ROOTS],
    *[root / "Unprocessed" / "M_AILABS_bona_fide_subset" for root in DATA_ROOTS],
    *[root / "Unprocessed" / "genuine_audio" for root in DATA_ROOTS],
])

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "svm"
SOURCE_MLP_MANIFESTS_DIR = PROJECT_ROOT / "outputs" / "mlp" / "manifests"
CACHE_DIR = OUTPUT_DIR / "cache"
MANIFESTS_DIR = OUTPUT_DIR / "manifests"
CONFIGS_DIR = OUTPUT_DIR / "configs"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
for directory in [OUTPUT_DIR, CACHE_DIR, MANIFESTS_DIR, CONFIGS_DIR, TABLE_DIR, FIGURE_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Synthetic audio:", SYNTHETIC_AUDIO_DIR)
print("Bona-fide audio:", BONA_FIDE_AUDIO_DIR)
print("SVM output directory:", OUTPUT_DIR)


Mounted at /content/drive
Google Drive mounted.
PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Education/INM701
Synthetic audio: /content/drive/MyDrive/Colab Notebooks/Education/INM701/Data/MLAAD_10pct/fake
Bona-fide audio: /content/drive/MyDrive/Colab Notebooks/Education/INM701/Data/genuine_audio
SVM output directory: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/svm


## 3. Frozen Train / Validation / Test Manifests


In [3]:
MANIFEST_COLUMNS = ["path", "relative_path", "label", "class_name", "language", "tts_generator", "source_file"]


def scan_audio_files(root, label):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Audio folder not found: {root}")
    rows = []
    for path in sorted(root.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in AUDIO_EXTENSIONS:
            continue
        relative_parts = path.relative_to(root).parts
        if label == 1 and len(relative_parts) >= 3 and relative_parts[0].lower() == "fake":
            language, tts_generator = relative_parts[1], relative_parts[2]
        elif label == 1 and len(relative_parts) >= 2:
            language, tts_generator = relative_parts[0], relative_parts[1]
        else:
            language = relative_parts[0] if len(relative_parts) >= 2 else "unknown"
            tts_generator = "bona_fide"
        rows.append({
            "path": str(path.resolve()),
            "relative_path": path.relative_to(root).as_posix(),
            "label": int(label),
            "class_name": CLASS_NAMES[int(label)],
            "language": language,
            "tts_generator": tts_generator,
            "source_file": path.stem,
        })
    return pd.DataFrame(rows, columns=MANIFEST_COLUMNS)


def split_config_matches(path):
    path = Path(path)
    if not path.exists():
        return False
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file).get("final_split_version") == FINAL_SPLIT_VERSION


split_config_path = MANIFESTS_DIR / "split_config.json"
train_manifest_path = MANIFESTS_DIR / "train_manifest.csv"
validation_manifest_path = MANIFESTS_DIR / "validation_manifest.csv"
test_manifest_path = MANIFESTS_DIR / "test_manifest.csv"
source_split_config = SOURCE_MLP_MANIFESTS_DIR / "split_config.json"
source_manifests = [SOURCE_MLP_MANIFESTS_DIR / name for name in ["train_manifest.csv", "validation_manifest.csv", "test_manifest.csv"]]

if split_config_matches(split_config_path) and all(path.exists() for path in [train_manifest_path, validation_manifest_path, test_manifest_path]):
    print("Using existing frozen SVM manifests:", MANIFESTS_DIR)
elif REUSE_MLP_SPLIT and split_config_matches(source_split_config) and all(path.exists() for path in source_manifests):
    for source_path in [source_split_config, *source_manifests]:
        shutil.copy2(source_path, MANIFESTS_DIR / source_path.name)
    print("Reused the exact frozen MLP split for SVM:", SOURCE_MLP_MANIFESTS_DIR)
elif RUN_FULL_SVM_DATA_PREPARATION:
    synthetic_manifest = scan_audio_files(SYNTHETIC_AUDIO_DIR, 1)
    bona_fide_manifest = scan_audio_files(BONA_FIDE_AUDIO_DIR, 0)
    if MAX_FILES_PER_CLASS is not None:
        synthetic_manifest = synthetic_manifest.sample(n=min(MAX_FILES_PER_CLASS, len(synthetic_manifest)), random_state=RANDOM_STATE)
        bona_fide_manifest = bona_fide_manifest.sample(n=min(MAX_FILES_PER_CLASS, len(bona_fide_manifest)), random_state=RANDOM_STATE)
    manifest = pd.concat([bona_fide_manifest, synthetic_manifest], ignore_index=True)
    manifest = manifest.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
    if manifest.empty:
        raise RuntimeError("No audio files were found.")

    train_manifest, remaining = train_test_split(
        manifest, train_size=0.70, random_state=RANDOM_STATE, stratify=manifest["label"]
    )
    validation_manifest, test_manifest = train_test_split(
        remaining, test_size=0.50, random_state=RANDOM_STATE, stratify=remaining["label"]
    )
    train_manifest = train_manifest.reset_index(drop=True)
    validation_manifest = validation_manifest.reset_index(drop=True)
    test_manifest = test_manifest.reset_index(drop=True)

    for left_name, left, right_name, right in [
        ("train", train_manifest, "validation", validation_manifest),
        ("train", train_manifest, "test", test_manifest),
        ("validation", validation_manifest, "test", test_manifest),
    ]:
        overlap = set(left["path"]) & set(right["path"])
        if overlap:
            raise RuntimeError(f"Path overlap between {left_name} and {right_name}: {len(overlap)}")

    train_manifest.to_csv(train_manifest_path, index=False)
    validation_manifest.to_csv(validation_manifest_path, index=False)
    test_manifest.to_csv(test_manifest_path, index=False)
    with open(split_config_path, "w", encoding="utf-8") as file:
        json.dump({
            "final_split_version": FINAL_SPLIT_VERSION,
            "random_state": RANDOM_STATE,
            "train_fraction": 0.70,
            "validation_fraction": 0.15,
            "test_fraction": 0.15,
            "creation_method": "deterministic label-stratified fallback; exact MLP manifest reuse is preferred",
        }, file, indent=2)
    print("Created a deterministic SVM split because the frozen MLP split was unavailable.")
else:
    raise FileNotFoundError(
        "Frozen manifests are missing. Run the MLP data-preparation notebook first, or set "
        "RUN_FULL_SVM_DATA_PREPARATION = True intentionally."
    )

train_manifest = pd.read_csv(train_manifest_path)
validation_manifest = pd.read_csv(validation_manifest_path)
test_manifest = pd.read_csv(test_manifest_path)

split_summary = pd.concat([
    train_manifest.assign(split="train"),
    validation_manifest.assign(split="validation"),
    test_manifest.assign(split="test"),
]).groupby(["split", "class_name"]).size().unstack(fill_value=0)
split_summary.to_csv(TABLE_DIR / "svm_split_summary.csv")
display(split_summary)


Reused the exact frozen MLP split for SVM: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/mlp/manifests


class_name,bona_fide,synthetic
split,,
test,600,1342
train,2798,6262
validation,599,1343


## 4. One-Time MFCC Extraction and Train-Only Scaling


In [4]:
def load_audio_fixed(path):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    if len(audio) > TARGET_SAMPLES:
        audio = audio[:TARGET_SAMPLES]
    elif len(audio) < TARGET_SAMPLES:
        audio = np.pad(audio, (0, TARGET_SAMPLES - len(audio)))
    peak = np.max(np.abs(audio)) if len(audio) else 0.0
    if peak > 0:
        audio = audio / peak
    return audio.astype(np.float32)


def extract_aggregated_mfcc(path):
    audio = load_audio_fixed(path)
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=SAMPLE_RATE,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window="hann",
        center=True,
    ).astype(np.float32)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)]).astype(np.float32)


def build_feature_table(split_manifest, split_name):
    features, labels, failures = [], [], []
    for row_index, row in split_manifest.reset_index(drop=True).iterrows():
        try:
            features.append(extract_aggregated_mfcc(row["path"]))
            labels.append(int(row["label"]))
        except Exception as exc:
            failures.append({"split": split_name, "row_index": int(row_index), "path": row["path"], "error": repr(exc)})
    if failures:
        failure_path = TABLE_DIR / f"svm_feature_extraction_failures_{split_name}.csv"
        pd.DataFrame(failures).to_csv(failure_path, index=False)
        raise RuntimeError(f"MFCC extraction failed for {len(failures)} {split_name} files. See {failure_path}")
    return np.vstack(features).astype(np.float32), np.asarray(labels, dtype=np.int64)


feature_config_path = CACHE_DIR / "feature_config.json"
cache_is_current = False
if feature_config_path.exists():
    with open(feature_config_path, "r", encoding="utf-8") as file:
        existing_config = json.load(file)
    cache_is_current = (
        existing_config.get("final_split_version") == FINAL_SPLIT_VERSION
        and existing_config.get("representation") == "aggregated MFCC mean+std (80 features), StandardScaler fitted on training only"
    )

if cache_is_current and all((CACHE_DIR / name).exists() for name in [
    "X_train.npy", "y_train.npy", "train_metadata.csv",
    "X_validation.npy", "y_validation.npy", "validation_metadata.csv",
    "X_test.npy", "y_test.npy", "test_metadata.csv", "svm_standard_scaler.joblib"
]):
    print("Existing final SVM cache is current:", CACHE_DIR)
elif RUN_FULL_SVM_DATA_PREPARATION:
    X_train_raw, y_train = build_feature_table(train_manifest, "train")
    X_validation_raw, y_validation = build_feature_table(validation_manifest, "validation")
    X_test_raw, y_test = build_feature_table(test_manifest, "test")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw).astype(np.float32)
    X_validation_scaled = scaler.transform(X_validation_raw).astype(np.float32)
    X_test_scaled = scaler.transform(X_test_raw).astype(np.float32)

    np.save(CACHE_DIR / "X_train.npy", X_train_scaled)
    np.save(CACHE_DIR / "y_train.npy", y_train)
    np.save(CACHE_DIR / "X_validation.npy", X_validation_scaled)
    np.save(CACHE_DIR / "y_validation.npy", y_validation)
    np.save(CACHE_DIR / "X_test.npy", X_test_scaled)
    np.save(CACHE_DIR / "y_test.npy", y_test)
    train_manifest.to_csv(CACHE_DIR / "train_metadata.csv", index=False)
    validation_manifest.to_csv(CACHE_DIR / "validation_metadata.csv", index=False)
    test_manifest.to_csv(CACHE_DIR / "test_metadata.csv", index=False)
    joblib.dump(scaler, CACHE_DIR / "svm_standard_scaler.joblib")

    feature_config = {
        "final_split_version": FINAL_SPLIT_VERSION,
        "representation": "aggregated MFCC mean+std (80 features), StandardScaler fitted on training only",
        "sample_rate": SAMPLE_RATE,
        "fixed_duration_seconds": FIXED_DURATION_SECONDS,
        "n_mfcc": N_MFCC,
        "n_fft": N_FFT,
        "hop_length": HOP_LENGTH,
        "win_length": WIN_LENGTH,
        "feature_dimension": int(X_train_scaled.shape[1]),
        "scaling": "StandardScaler fitted on raw X_train only; same fitted transform applied to validation and test",
        "test_features_prepared_but_not_evaluated": True,
    }
    with open(feature_config_path, "w", encoding="utf-8") as file:
        json.dump(feature_config, file, indent=2)
    print("Saved shared SVM cache:", CACHE_DIR)
else:
    raise RuntimeError("SVM cache is missing or outdated and full preparation is disabled.")


Saved shared SVM cache: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/svm/cache
